# Первый вариант

In [ ]:
# =============================================================================
# УНИВЕРСАЛЬНЫЙ KAGGLE TABULAR TEMPLATE 2025 (CatBoost + LightGBM + XGBoost + NN + Stacking)
# Автор: адаптация топ-1–10 решений 2023–2025
# Что даёт почти всегда топ-1% после хорошего FE
# =============================================================================

! pip install --quiet catboost lightgbm xgboost optuna scikit-learn-intelex pandas numpy category_encoders

import pandas as pd
import numpy as np
import warnings, os, gc, sys
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, KFold, RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, mean_squared_error, log_loss
from sklearn.preprocessing import StandardScaler, LabelEncoder
import category_encoders as ce

import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from catboost import Pool, CatBoostClassifier, CatBoostRegressor

import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# =============================================================================
# КОНФИГУРАЦИЯ — МЕНЯЕШЬ ТОЛЬКО ЗДЕСЬ
# =============================================================================

DATA_PATH = Path('/kaggle/input/your-competition-name')  # поменяй
train = pd.read_csv(DATA_PATH / 'train.csv')
test  = pd.read_csv(DATA_PATH / 'test.csv')
ss    = pd.read_csv(DATA_PATH / 'sample_submission.csv')

TARGET = 'target'                    # название целевой переменной
ID_COL = 'id'                        # если есть
TASK   = 'classification'            # 'classification' или 'regression'

N_FOLDS = 5 or 10                    # 10 почти всегда лучше, если времени хватает
SEED    = 42

# Если уже есть крутые фичи — просто добавь их в train/test ниже
# =============================================================================

def seed_everything(seed=42):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)

# =============================================================================
# Простейший Feature Engineering (добавляй сюда всё что найдёшь)
# =============================================================================

def feature_engineering(df):
    df = df.copy()
    
    # Пример базовых фич (удали/добавь свои)
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if TARGET in num_cols: num_cols.remove(TARGET)
    if ID_COL in num_cols: num_cols.remove(ID_COL)
    
    # статистики по группам, взаимодействия и т.д. — сюда вставляешь свои лучшие фичи
    # df['mean_per_some_cat'] = df.groupby('cat')['num'].transform('mean')
    
    return df

train = feature_engineering(train)
test  = feature_engineering(test)

features = [c for c in train.columns if c not in [TARGET, ID_COL]]

cat_features = train[features].select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_features:
    if train[col].dtype == 'object':
        le = LabelEncoder()
        train[col] = le.fit_transform(train[col].astype(str))
        test[col]  = le.transform(test[col].astype(str))

# =============================================================================
# 1. CatBoost baseline (часто уже топ-5% сам по себе)
# =============================================================================

cat_params = {
    'loss_function'  : 'Logloss' if TASK == 'classification' else 'RMSE',
    'eval_metric'    : 'AUC' if TASK == 'classification' else 'RMSE',
    'depth'          : 10,
    'learning_rate' : 0.05,
    'random_seed'    : SEED,
    'task_type'      : 'GPU' if os.path.exists('/usr/local/cuda') else 'CPU',
    'devices'        : '0:1:2:3',
    'iterations'     : 5000,
    'early_stopping_rounds': 200,
    'verbose'        : 500,
    'thread_count'   : -1,
    'l2_leaf_reg'    : 3,
    'bagging_temperature': 0.8,
    'random_strength': 1.5,
}

def fit_catboost(train_X, train_y, val_X, val_y, test_X, params, cat_features):
    model = cb.CatBoost(params)
    model.fit(train_X, train_y,
               eval_set=(val_X, val_y),
               cat_features=cat_features,
               use_best_model=True,
               verbose=500)
    
    pred_val = model.predict_proba(val_X)[:,1] if TASK == 'classification' else model.predict(val_X)
    pred_test = model.predict_proba(test_X)[:,1] if TASK == 'classification' else model.predict(test_X)
    return model, pred_val, pred_test

# =============================================================================
# 2. LightGBM + XGBoost (ещё 2 сильные модели)
# =============================================================================

lgb_params = {
    'objective'     : 'binary' if TASK == 'classification' else 'regression',
    'metric'        : 'auc' if TASK == 'classification' else 'rmse',
    'learning_rate' : 0.05,
    'num_leaves'    : 256,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq'  : 5,
    'seed'          : SEED,
    'verbose'       : -1,
    'device'        : 'gpu' if os.path.exists('/usr/local/cuda') else 'cpu',
    'max_bin'       : 255,
}

xgb_params = {
    'objective'         : 'binary:logitraw' if TASK == 'classification' else 'reg:squarederror',
    'eval_metric'       : 'auc' if TASK == 'classification' else 'rmse',
    'learning_rate'     : 0.05,
    'max_depth'         : 10,
    'subsample'         : 0.8,
    'colsample_bytree'  : 0.8,
    'tree_method'       : 'gpu_hist' if os.path.exists('/usr/local/cuda') else 'hist',
    'random_state'      : SEED,
    'verbosity'         : 0,
}

# =============================================================================
# OOF + Stacking
# =============================================================================

oof_cat = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

pred_cat = np.zeros(len(test))
pred_lgb = np.zeros(len(test))
pred_xgb = np.zeros(len(test))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED) if TASK == 'classification' else KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

metric = roc_auc_score if TASK == 'classification' else mean_squared_error

scores = []

for fold, (trn_idx, val_idx) in enumerate(skf.split(train, train[TARGET] if TASK == 'classification' else None)):
    print(f"\nFold {fold+1}/{N_FOLDS}")
    X_train, y_train = train.iloc[trn_idx][features], train.iloc[trn_idx][TARGET]
    X_val,   y_val   = train.iloc[val_idx][features],   train.iloc[val_idx][TARGET]
    X_test = test[features].copy()
    
    # CatBoost
    model_cat, val_cat, test_cat = fit_catboost(X_train, y_train, X_val, y_val, X_test, cat_params, cat_features)
    oof_cat[val_idx] = val_cat
    pred_cat += test_cat / N_FOLDS
    
    # LightGBM
    lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=cat_features)
    lgb_valid = lgb.Dataset(X_val, y_val, categorical_feature=cat_features, reference=lgb_train)
    model_lgb = lgb.train(lgb_params, lgb_train, valid_sets=[lgb_valid],
                          num_boost_round=10000,
                          callbacks=[lgb.early_stopping(200), lgb.log_evaluation(500)])
    
    val_lgb = model_lgb.predict(X_val)
    if TASK == 'classification':
        val_lgb = 1/(1+np.exp(-val_lgb))  # sigmoid
    oof_lgb[val_idx] = val_lgb
    pred_lgb += val_lgb / N_FOLDS
    
    # XGBoost
    model_xgb = xgb.XGBModel(**xgb_params)
    model_xgb.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  early_stopping_rounds=200,
                  verbose=500)
    
    val_xgb = model_xgb.predict(X_val)
    if TASK == 'classification':
        val_xgb = 1/(1+np.exp(-val_xgb))
    oof_xgb[val_idx] = val_xgb
    pred_xgb += val_xgb / N_FOLDS
    
    # текущий скор по фолду
    if TASK == 'classification':
        score = roc_auc_score(y_val, 0.4*val_cat + 0.3*val_lgb + 0.3*val_xgb)
    else:
        score = mean_squared_error(y_val, 0.4*val_cat + 0.3*val_lgb + 0.3*val_xgb, squared=False)
    
    scores.append(score)
    print(f"Fold {fold+1} blend AUC/RMSE: {score:.6f}")

print(f"\nCV mean: {np.mean(scores):.6f} ± {np.std(scores):.6f}")

# =============================================================================
# Финальный бленд (можно ещё Optuna подобрать веса)
# =============================================================================

if TASK == 'classification':
    final_pred = 0.45 * pred_cat + 0.30 * pred_lgb + 0.25 * pred_xgb
else:
    final_pred = 0.45 * pred_cat + 0.30 * pred_lgb + 0.25 * pred_xgb

ss[TARGET] = final_pred
ss.to_csv('submission_stacking.csv', index=False)
print("Submission saved!")

# Если хочешь ещё выше — добавь второй уровень стекинга (logistic regression на oof)

# Второй с оптюной и табнетом

In [ ]:
!pip install --quiet catboost lightgbm xgboost optuna pytorch-tabnet torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

import pandas as pd, numpy as np, gc, warnings, os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb, catboost as cb, xgboost as xgb
import optuna, torch
from pytorch_tabnet.tab_model import TabNetClassifier
warnings.filterwarnings('ignore')

# ====================== КОНФИГ ======================
DATA_DIR = '/kaggle/input/your-competition-name-here'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
ss = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
TARGET = 'target'
ID_COL = 'id'
TASK = 'classification'   # или 'regression'
N_FOLDS = 10
SEED = 42
OPTUNA_TRIALS = 150

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(SEED)

# ====================== УМНЫЙ FE ======================
def smart_fe(df):
    df = df.copy()
    
    # 1. Только самые важные аггрегации (выбирай вручную топ-категории по важности!)
    important_cats = ['cat_feature1', 'cat_feature2']  # ←←← ЗАМЕНИ на свои топ-категории (по gain из первой модели)
    num_cols = df.select_dtypes(include='number').columns.drop([TARGET, ID_COL], errors='ignore')
    
    for col in important_cats:
        if col not in df.columns: continue
        for stat in ['mean', 'std', 'min', 'max']:
            for num in num_cols[:15]:  # только 15 числовых
                map_dict = df.groupby(col)[num].agg(stat).to_dict()
                df[f'{col}_{num}_{stat}'] = df[col].map(map_dict)
    
    # 2. Частотный энкодинг (всегда полезно)
    for col in df.select_dtypes('object').columns:
        freq = df[col].value_counts()
        df[f'{col}_freq'] = df[col].map(freq)
    
    return df

train = smart_fe(train)
test = smart_fe(test)

features = [c for c in train.columns if c not in [TARGET, ID_COL]]
cat_features_indices = [i for i, c in enumerate(features) if train[c].dtype == 'object' or train[c].nunique() < 200]

print(f"Total features: {len(features)}")

# ====================== OOF ======================
oofs = {'cat': np.zeros(len(train)), 'lgb': np.zeros(len(train)), 
        'xgb': np.zeros(len(train)), 'tab': np.zeros(len(train))}
preds = {k: np.zeros(len(test)) for k in oofs}
scores = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (trn_idx, val_idx) in enumerate(skf.split(train, train[TARGET])):
    print(f"\nFOLD {fold+1}/{N_FOLDS}")
    X_tr, y_tr = train.iloc[trn_idx][features], train.iloc[trn_idx][TARGET]
    X_val, y_val = train.iloc[val_idx][features], train.iloc[val_idx][TARGET]
    X_test = test[features]

    # 1. CatBoost (самый сильный в 2025)
    cat_params = {
        'iterations': 3000,
        'learning_rate': 0.03,
        'depth': 8,                    # ↓↓↓ было 10
        'l2_leaf_reg': 10,
        'random_strength': 0.8,
        'bagging_temperature': 0.2,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'random_seed': SEED,
        'task_type': 'GPU' if torch.cuda.is_available() else 'CPU',
        'devices': '0',
        'early_stopping_rounds': 300,
        'verbose': 500
    }
    model = cb.CatBoost(cat_params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), 
              cat_features=cat_features_indices, use_best_model=True)
    oofs['cat'][val_idx] = model.predict_proba(X_val)[:,1]
    preds['cat'] += model.predict_proba(X_test)[:,1] / N_FOLDS

    # 2. LightGBM (categorical_feature вместо LabelEncoder!)
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'learning_rate': 0.03,
        'num_leaves': 128,             # ↓↓↓ было 256
        'feature_fraction': 0.7,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'seed': SEED,
        'verbose': -1,
        'device': 'gpu',
        'gpu_platform_id': 0,
        'gpu_device_id': 0,
        'deterministic': True
    }
    lgb_train = lgb.Dataset(X_tr, y_tr, categorical_feature=cat_features_indices)
    lgb_valid = lgb.Dataset(X_val, y_val, reference=lgb_train, categorical_feature=cat_features_indices)
    model_lgb = lgb.train(lgb_params, lgb_train, valid_sets=[lgb_valid],
                          num_boost_round=5000, callbacks=[lgb.early_stopping(300), lgb.log_evaluation(0)])
    val_pred = model_lgb.predict(X_val)
    oofs['lgb'][val_idx] = val_pred
    preds['lgb'] += model_lgb.predict(X_test) / N_FOLDS

    # 3. XGBoost — можно вообще убрать, он редко даёт прирост в 2025
    # 4. TabNet — оставляем только если датасет < 300k строк, иначе слишком долго

    # Бленд на этом фолде
    blend = 0.55 * oofs['cat'][val_idx] + 0.45 * oofs['lgb'][val_idx]
    score = roc_auc_score(y_val, blend)
    scores.append(score)
    print(f"Fold {fold+1} blend AUC: {score:.6f}")

print(f"\nCV before Optuna: {np.mean(scores):.6f} ± {np.std(scores):.4f}")

# ====================== Optuna весов ======================
def objective(trial):
    w_cat = trial.suggest_float('cat', 0.3, 0.8)
    w_lgb = trial.suggest_float('lgb', 0.2, 0.7)
    total = w_cat + w_lgb
    blend = (w_cat * (oofs['cat']) + w_lgb * (oofs['lgb'])) / total
    return roc_auc_score(train[TARGET], blend)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=OPTUNA_TRIALS)
best = study.best_params
total = sum(best.values())
weights = {k: v/total for k,v in best.items()}
print(f"Best weights: {weights} | Final CV: {study.best_value:.6f}")

# Финальный сабмит
final_pred = (weights['cat'] * preds['cat'] + weights['lgb'] * preds['lgb'])
ss[TARGET] = final_pred
ss.to_csv('submission_2025_real.csv', index=False)